# 11주차 랩 · 시계열 예측

**⏱ 60분** · 한국관광공사 국적별 방한 외래객

## 🎯 이번 랩의 질문

교재에서는 미국 항공 여객을 여러 방법으로 예측해 보고, 테스트 기간의 오차가 가장 작은 방법을 골랐다. 이번에는 같은 일을 한국에 들어온 외국인 방문객으로 해 본다. 각자 배정받은 나라 하나의 월별 방문객 자료를 직접 내려받아 정리하고, 2019년 한 해를 테스트 기간으로 두어 교재의 여섯 방법 가운데 어느 방법이 가장 잘 맞히는지 찾는다. 나라마다 방문객의 추세와 계절 모양이 다르므로 답도 사람마다 다르다.

> **배정받은 나라의 월별 방한객은 어떤 방법으로 예측하는 것이 가장 정확한가?**

코드는 교재의 셀을 가져와 이번 자료에 맞게 고쳐 쓴다. 다만 엑셀 파일을 읽거나 표의 모양을 바꾸는 일처럼 교재에 나오지 않은 작업도 있는데, 그런 작업은 방법을 직접 찾아서 한다.

### 제출

이 노트북과 1절에서 내려받은 엑셀 파일을 함께 LMS에 제출한다. 노트북은 Colab의 **파일 → 다운로드 → .ipynb 다운로드**로 받고, 엑셀 파일은 이름을 바꾸지 않고 그대로 낸다. 채점할 때는 제출한 엑셀 파일로 노트북을 처음부터 다시 실행하므로, 엑셀 파일이 없으면 답을 채점할 수 없다. 엑셀 파일은 내려받을 때마다 파일이 달라져 누가 받은 것인지 구별되므로, 다른 사람이 받은 파일을 쓰지 않는다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행
import numpy as np
import pandas as pd
pd.set_option("display.max_rows", 500)       # 표를 줄임표(...) 없이 끝까지 보여 줌
pd.set_option("display.max_columns", 50)
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import checker, summary, check_format
check = checker("week11")
print("준비 완료!")

---
## 1. 자료 내려받아 읽기

먼저 분석할 자료를 관광지식정보시스템에서 직접 내려받는다. 2010년부터 2019년까지 나라별로 매달 몇 명이 한국에 들어왔는지를 담은 표이다.

1. [관광지식정보시스템 입국관광통계](https://know.tour.go.kr/stat/entryTourStatDis19Re.do)에 들어간다.
2. 통계 유형에서 **국적별 입국**, 기간 구분에서 **월별**을 고르고, 기간을 **2010년 01월**부터 **2019년 12월**까지로 둔 뒤 **조회**를 누른다.
3. **엑셀내려받기** → **내려받기(셀병합 해제)**를 누르면 `국적별 입국_숫자.xls` 파일이 내려받아진다.
4. Colab 왼쪽의 📁 파일 창에 이 파일을 끌어다 놓아 올린다. 런타임이 다시 시작되면 올린 파일이 지워지므로 다시 올린다.

내려받은 파일을 읽어 어떻게 생긴 표인지부터 본다. 엑셀 파일을 읽는 방법은 교재에 없으므로 직접 찾는다. 배정받은 나라 이름은 띄어쓰기 없이 쓴다(예: "일본", "몽골").

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
country = None  # 배정받은 나라 이름(띄어쓰기 없이)
raw = None  # 내려받은 엑셀 파일을 읽은 표

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 내려받은 표의 앞 다섯 행이 나온다. 한 행이 한 나라이고, 오른쪽으로 달마다 방한객 수가 이어진다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
raw.head()

---
## 2. 배정받은 나라의 월별 표로 정리하기

내려받은 표는 한 행이 한 나라라서, 교재처럼 한 행이 한 달인 시계열과 모양이 다르다. 계산에 쓰지 않는 행과 열도 섞여 있고, 나라 이름이 화면과 다르게 적힌 것도 있다. 그래서 이 표에서 배정받은 나라만 골라, 교재 1절의 표처럼 한 행이 한 달이고 날짜가 index이며 방문객 수가 숫자로 든 열 하나만 있는 표로 바꾼다. 무엇을 빼고 바꿔야 하는지는 표를 직접 보고 찾고, 교재에 없는 기능은 방법을 직접 찾는다.

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
df = None  # 한 행이 한 달, index가 날짜, 열은 방한객 수 visitors 하나인 표

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 `(120, 1)`이 나온다. 10년 120개월이 행이고 방한객 수 열 하나만 있다는 뜻이다. 그 아래에는 2010년부터 2019년까지 배정받은 나라의 월별 방한객이 선 그래프로 그려진다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
print(df.shape)
df["visitors"].plot(figsize=(9, 3.5))
plt.show()

---
## 3. 2019년을 예측해 비교하기

이제 교재 2절부터 6절까지의 절차를 그대로 밟는다. 마지막 12개월인 2019년을 테스트 기간, 그 앞 9년을 학습 기간으로 나누고, 교재의 여섯 방법으로 2019년을 예측한 뒤 MAPE로 비교한다. 채점에서 방법 이름으로 답을 맞춰 보므로, 방법 이름은 교재와 같게 쓴다.

| 교재 | 방법 이름 |
|---|---|
| 3절 | `naive`, `seasonal naive` |
| 4절 | `AR(4)`, `seasonal AR` |
| 5절 | `linear`, `linear + season` |

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
train = None  # 학습 기간: 2010년 1월부터 2018년 12월까지
test = None  # 테스트 기간: 2019년 12개월
y_test = None  # 테스트 기간의 실제 방한객 수
preds = {}  # 방법 이름: 테스트 기간의 예측값
fits = {}  # 방법 이름: 학습 기간의 적합값
preds["naive"] = None
preds["seasonal naive"] = None

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 학습 기간과 테스트 기간의 길이 `108 12`가 나오고, 2019년 1월부터 12월까지의 naive 예측과 seasonal naive 예측이 두 열로 나온다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
print(len(train), len(test))
pd.DataFrame(preds).round(0)

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
preds["AR(4)"] = None
fits["AR(4)"] = None
preds["seasonal AR"] = None
fits["seasonal AR"] = None

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 앞의 두 열에 `AR(4)`와 `seasonal AR`이 더해져 2019년 예측값이 네 열로 나온다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
pd.DataFrame(preds).round(0)

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
preds["linear"] = None
fits["linear"] = None
preds["linear + season"] = None
fits["linear + season"] = None

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 `linear`와 `linear + season`이 더해져 여섯 방법의 2019년 예측값이 여섯 열로 나온다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
pd.DataFrame(preds).round(0)

In [ ]:
# 📝 None 자리를 채운다. 셀은 필요한 만큼 추가한다.
table = None  # 방법마다 테스트 기간의 MSE·MAPE를 모은 표
mape = None  # 적합값이 있는 방법의 학습 MAPE와 테스트 MAPE를 나란히 놓은 표

위 셀을 제대로 채웠다면 아래 셀을 실행했을 때 여섯 방법의 MSE와 MAPE가 MAPE가 작은 순서로 나오고, 그 아래에 적합값이 있는 네 방법의 학습 MAPE와 테스트 MAPE가 나란히 나온다.

In [ ]:
# ▶ 그대로 실행 — 결과 확인
print(table.sort_values("MAPE").round(2))
print(mape.round(2))

---
## 4. 답안

앞에서 만든 표에서 답 네 개를 꺼내 아래 변수에 담는다. 채점은 노트북을 다시 실행해 이 변수의 값을 보므로, 숫자를 손으로 옮겨 적지 말고 코드로 꺼내고 반올림하지 않는다. MAPE가 가장 작은 방법의 이름을 코드로 꺼내는 방법은 직접 찾는다.

In [ ]:
# 📝 답안 — None 자리를 채워라
total_visitors = None   # 2010년 1월 ~ 2019년 12월 방한객 합계(명)
snaive_mape = None      # seasonal naive 예측의 2019년 MAPE(%)
best = None             # 2019년 MAPE가 가장 작은 방법의 이름(3단계 표의 이름 그대로)
best_mape = None        # 그 방법의 2019년 MAPE(%)

In [ ]:
# ✅ 확인 — 답안 (고치지 말고 실행만)
if os.environ.get("BALAB_ANSWER_KEY"):
    check(1, total_visitors=total_visitors)
    check(2, snaive_mape=(snaive_mape, 0.01))
    check(3, best=best)
    check(4, best_mape=(best_mape, 0.01))
else:
    check_format(1, total_visitors=total_visitors)
    check_format(2, snaive_mape=snaive_mape)
    check_format(3, best=best)
    check_format(4, best_mape=best_mape)

---
## 5. 해석

마지막으로 본인 결과의 그림과 표를 읽고 짧게 답한다. 이 부분은 자동 채점하지 않는다.

1. 2절의 그림에서 해마다 방한객이 가장 많은 달은 대개 몇 월인가? 10년 동안 늘었나, 줄었나?
2. 학습 기간 MAPE가 가장 작은 방법과 2019년 MAPE가 가장 작은 방법은 같은가? 다음 해를 예측한다면 어느 방법을 쓰겠나? 표의 숫자 하나로 근거를 적어라.

✏️ **나의 답:** (여기를 두 번 눌러 적는다)

---
## ✅ 제출 전 확인

제출하기 전에 아래 셀을 실행해 답안이 모두 채워졌고 형식이 맞는지 확인한다. "형식 확인 완료"는 형식만 본 결과이고 값이 맞다는 뜻은 아니다. 해석까지 적었으면 이 노트북과 내려받은 엑셀 파일을 함께 제출한다.

In [ ]:
# ✅ 제출 전 확인 — 답안의 작성 여부와 형식
summary()